# Activity 2: Can multimodal data predict tomorrow's fatigue?

[Open this notebook in Google Colab](https://colab.research.google.com/github/sonya-j/ninr-ai-bootcamp/blob/main/notebooks/02_multimodal_symptom_trajectories.ipynb)

**Beginner-friendly goal:** Use real longitudinal data to compare a simple symptom baseline with models that add wearable signals and time context.

## 1. Start with a question the data can answer

The broad research question asks whether patient reports, wearables, EHR data, and context can predict symptom trajectories. This public dataset contains **patient reports, wearable measurements, and time context—but no EHR data**.

> **Activity question:** Can information available today predict a participant's 1–10 fatigue rating on the next calendar day?

The prediction is made at the end of today. Tomorrow's fatigue is the outcome and must never be used as an input.

## 2. Meet the public dataset

The original investigators released *Continuous multi-sensor wearable data and daily subject-reported fatigue of healthy adults*, version 1, on [Zenodo](https://doi.org/10.5281/zenodo.4266157) under CC BY 4.0. The [study article](https://pmc.ncbi.nlm.nih.gov/articles/PMC7768149/) describes the collection and analyses.

Important: these were **healthy adult participants**, not a clinical patient cohort. The repository contains a documented daily preparation of the original one-minute sensor files and daily questionnaires.

In [ ]:
from pathlib import Path
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
local_path = Path('../data/processed/multimodal_symptom_trajectories/participant_day.csv')
github_url = 'https://raw.githubusercontent.com/sonya-j/ninr-ai-bootcamp/main/data/processed/multimodal_symptom_trajectories/participant_day.csv'
data = pd.read_csv(local_path if local_path.exists() else github_url, parse_dates=['date'])
print(f'{len(data):,} participant-days from {data.participant_id.nunique()} participants')
print(f'{data.fatigue_tomorrow.notna().sum():,} rows have an immediately following next-day outcome')
data.head(3)

## 3. Audit the modalities before modeling

A multimodal model combines different kinds of information. An absent modality cannot be evaluated.

In [ ]:
modality_audit = pd.DataFrame({
    'modality': ['Patient-reported', 'Wearable', 'Time context', 'EHR', 'Outcome'],
    'available': ['Yes', 'Yes', 'Yes', 'NO', 'Yes'],
    'examples': [
        'fatigue, physical/mental exhaustion, sport',
        'activity, heart rate, HRV, respiration, steps, skin temperature',
        'study day, day of week, weekend',
        'No diagnoses, medications, encounters, labs, or notes',
        'next-calendar-day fatigue rating (1–10)',
    ],
})
display(modality_audit)

## 4. Explore one participant's trajectory

Choose a participant. Look for changes, gaps, and whether heart rate appears to move with fatigue. A pattern in one person is useful for generating questions, but it is not proof of prediction.

In [ ]:
participant_picker = widgets.Dropdown(
    options=sorted(data.participant_id.unique()),
    description='Participant:',
    value=int(data.groupby('participant_id').size().idxmax()),
)

def plot_participant(participant):
    person = data[data.participant_id == participant].sort_values('date')
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(person.date, person.fatigue_today, marker='o', label="Today's fatigue")
    axes[0].plot(person.date, person.fatigue_tomorrow, marker='s', linestyle='--', label="Tomorrow's fatigue")
    axes[0].set_ylabel('Fatigue (1–10)')
    axes[0].set_ylim(0.5, 10.5)
    axes[0].legend()
    axes[1].plot(person.date, person.hr_mean, marker='o', color='#B35C1E')
    axes[1].set_ylabel('Daily mean heart rate')
    axes[1].set_xlabel('Date')
    fig.suptitle(f'Participant {participant}: {len(person)} matched days')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

display(participant_picker)
trajectory_output = widgets.interactive_output(plot_participant, {'participant': participant_picker})
display(trajectory_output)

## 5. Inspect missingness

Missing wearable values are not the same as zero. They may reflect non-wear, device limitations, or technical failure. Missing next-day fatigue usually means there was no report on the immediately following calendar day.

In [ ]:
missing = data.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]
ax = missing.sort_values().plot.barh(figsize=(9, 5), color='#3F7CAC')
ax.set_xlabel('Missing observations (%)')
ax.set_ylabel('Variable')
ax.set_title('Missingness in the prepared participant-day data')
plt.tight_layout()
plt.show()

## 6. Create an honest train/test split

We keep each person entirely in training or testing. A random row split would let the model see other days from the same participant and could make performance look better than it is for a new person.

In [ ]:
model_data = data.dropna(subset=['fatigue_tomorrow']).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2027)
train_index, test_index = next(splitter.split(model_data, groups=model_data.participant_id))
train = model_data.iloc[train_index].copy()
test = model_data.iloc[test_index].copy()
print(f'Training: {len(train)} days, {train.participant_id.nunique()} participants')
print(f'Testing:  {len(test)} days, {test.participant_id.nunique()} participants')
assert set(train.participant_id).isdisjoint(set(test.participant_id))

## 7. Compare feature sets

The first model is intentionally simple: use today's fatigue to predict tomorrow's. The other models must beat this persistence baseline to justify collecting more information.

In [ ]:
patient_reported = ['fatigue_today', 'change_vs_yesterday', 'physical_exhaustion', 'mental_exhaustion', 'sport_today']
wearable = ['wear_minutes'] + [c for c in data.columns if c.endswith('_mean') or c.endswith('_std')]
context = ['study_day', 'day_of_week', 'weekend']
feature_sets = {
    "Today's fatigue only": ['fatigue_today'],
    'Wearable only': wearable,
    'Patient report + wearable': patient_reported + wearable,
    'All available (no EHR)': patient_reported + wearable + context,
}
pd.DataFrame({'feature set': feature_sets.keys(), 'number of variables': [len(v) for v in feature_sets.values()]})

In [ ]:
def make_pipeline(features, model_name):
    categorical = [c for c in features if not pd.api.types.is_numeric_dtype(train[c])]
    numeric = [c for c in features if c not in categorical]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    prep = ColumnTransformer([('numeric', numeric_pipe, numeric), ('categorical', categorical_pipe, categorical)])
    model = Ridge(alpha=1.0) if model_name == 'Ridge regression' else RandomForestRegressor(
        n_estimators=300, min_samples_leaf=4, random_state=2027, n_jobs=1
    )
    return Pipeline([('prepare', prep), ('model', model)])

def fit_and_score(feature_name, model_name):
    features = feature_sets[feature_name]
    pipeline = make_pipeline(features, model_name)
    pipeline.fit(train[features], train.fatigue_tomorrow)
    predictions = np.clip(pipeline.predict(test[features]), 1, 10)
    result = test[['participant_id', 'date', 'fatigue_tomorrow']].copy()
    result['prediction'] = predictions
    result['absolute_error'] = (result.fatigue_tomorrow - result.prediction).abs()
    return pipeline, result

## 8. Interactive model comparison

Choose a feature set and model. Lower MAE is better. R² can be negative when a model performs worse than predicting a constant mean.

In [ ]:
feature_picker = widgets.Dropdown(options=list(feature_sets), value='All available (no EHR)', description='Inputs:')
model_picker = widgets.RadioButtons(options=['Ridge regression', 'Random forest'], value='Ridge regression', description='Model:')

def show_model(feature_name, model_name):
    _, result = fit_and_score(feature_name, model_name)
    mae = mean_absolute_error(result.fatigue_tomorrow, result.prediction)
    r2 = r2_score(result.fatigue_tomorrow, result.prediction)
    display(Markdown(f'### Test result: MAE **{mae:.2f} fatigue points** · R² **{r2:.2f}**'))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].scatter(result.fatigue_tomorrow, result.prediction, alpha=.75)
    axes[0].plot([1, 10], [1, 10], linestyle='--', color='black')
    axes[0].set(xlabel='Observed tomorrow fatigue', ylabel='Predicted tomorrow fatigue', xlim=(0.5, 10.5), ylim=(0.5, 10.5))
    person_mae = result.groupby('participant_id').absolute_error.mean().sort_values()
    person_mae.plot.barh(ax=axes[1], color='#3F7CAC')
    axes[1].set(xlabel='MAE (fatigue points)', ylabel='Test participant')
    fig.suptitle(f'{feature_name} · {model_name}')
    plt.tight_layout()
    plt.show()

display(widgets.HBox([feature_picker, model_picker]))
model_output = widgets.interactive_output(show_model, {'feature_name': feature_picker, 'model_name': model_picker})
display(model_output)

## 9. Put every comparison in one table

A single selected result can be misleading. This cell evaluates all eight combinations on the same held-out participants.

In [ ]:
rows = []
for model_name in ['Ridge regression', 'Random forest']:
    for feature_name in feature_sets:
        _, result = fit_and_score(feature_name, model_name)
        rows.append({
            'model': model_name,
            'inputs': feature_name,
            'MAE': mean_absolute_error(result.fatigue_tomorrow, result.prediction),
            'R²': r2_score(result.fatigue_tomorrow, result.prediction),
        })
comparison = pd.DataFrame(rows).sort_values('MAE')
comparison['MAE'] = comparison['MAE'].round(2)
comparison['R²'] = comparison['R²'].round(2)
display(comparison.reset_index(drop=True))

## 10. Write a calibrated conclusion

Complete this statement using your results:

> In this small sample of healthy adults, a model using **_____** had an MAE of **_____ fatigue points** for next-day fatigue among held-out participants. Compared with the today's-fatigue baseline, performance was **_____**. This does not establish **_____**.

Check your wording:

- Say **association/prediction**, not cause.
- Name the population: healthy adults in this pilot dataset.
- Do not claim EHR benefit; EHR data were absent.
- Do not claim clinical readiness from a small retrospective exercise.

## 11. Design—not fabricate—the EHR extension

Choose one real EHR field that might add useful information: diagnoses, medication changes, laboratory results available before the prediction time, recent encounters, or treatment exposure. Discuss:

1. Why might it predict tomorrow's fatigue?
2. When was it recorded, and is it available by the end-of-day cutoff?
3. How would you align it to a daily wearable/report timeline?
4. Whose EHR data could be incomplete?
5. What nursing action would a prediction support?

**Key lesson:** The public data support a three-modality test. Answering the full four-modality question requires a separately governed linked dataset and a new validation study.